# Lesson 1: Simple ReAct Agent from Scratch

In [ ]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [1]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [2]:
client =  OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),  # 从环境变量读取
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

In [3]:
chat_completion = client.chat.completions.create(
    model="qwen-max",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [4]:
chat_completion.choices[0].message.content

"Hello! It's nice to meet you. How can I assist you today?"

In [5]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="qwen-max", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [6]:
# 提示词中文翻译如下
# 你陷入了思考、行动、暂停、观察的循环中。
# 循环结束时，输出答案。
# 用“思考”一词来描述你对所提问题的想法。
# 使用操作来运行可用的操作之一 - 然后返回 PAUSE。
# 观察结果将是执行这些操作的结果。

# 您可以采取以下行动：

# calculate:
# 例如，计算：4 * 7 / 3
# 运行计算并返回结果——使用 Python，因此必要时请确保使用浮点语法。

# average_dog_weight:
# 例如：average_dog_weight: Collie
# 返回给定品种的狗的平均体重

# 示例会话：

# 问：斗牛犬有多重？
# 想法：我应该用 average_dog_weight 来查看狗狗的体重。
# 操作：平均狗体重：斗牛犬
# 暂停

# 您将再次接到以下电话：

# 观察：一只斗牛犬重 51 磅。

# 然后输出：

# 答案：斗牛犬重 51 磅。
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [8]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [9]:
abot = Agent(prompt)

In [10]:
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I should look up the average weight of a toy poodle using the average_dog_weight action.
Action: average_dog_weight: Toy Poodle
PAUSE


In [11]:
result = average_dog_weight("Toy Poodle")

In [12]:
result

'a toy poodles average weight is 7 lbs'

In [13]:
next_prompt = "Observation: {}".format(result)

In [14]:
abot(next_prompt)

"Answer: A toy poodle's average weight is 7 lbs."

In [15]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [16]:
abot = Agent(prompt)

In [17]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: To find the combined weight of the two dogs, I need to first determine the average weight of each breed and then sum them up. I will start by finding the average weight of a Border Collie using the `average_dog_weight` action.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [18]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [19]:
abot(next_prompt)

'Thought: Now that I have the average weight of a Border Collie, which is 37 lbs, I need to find the average weight of a Scottish Terrier. I will use the `average_dog_weight` action again.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [20]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [21]:
abot(next_prompt)

'Thought: I now have the average weights for both breeds: a Border Collie at 37 lbs and a Scottish Terrier at 20 lbs. To find the combined weight, I just need to add these two values together.\nAction: calculate: 37 + 20\nPAUSE'

In [22]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [23]:
abot(next_prompt)

'Answer: The combined weight of a Border Collie and a Scottish Terrier is 57 lbs.'

### Add loop 

In [26]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [27]:
def query(question, max_turns=5):
    """
    这个函数接收一个'question'（你的初始问题），
    然后让一个 AI 代理循环'max_turns'次（默认5次）来尝试解答它。
    """
    # i 是一个计数器，用来追踪我们进行了多少轮“思考->行动”循环。
    i = 0
    
    # 根据一个预设的'prompt'（指示）来初始化我们的 AI 代理（bot）。
    # 这就像给了 AI 一份“说明书”，告诉它该怎么做。
    bot = Agent(prompt)
    # 'next_prompt' 存储着下一次要发送给 AI 的内容。
    # 一开始，这个内容就是用户最初提的 'question'。
    next_prompt = question
    # 开始主循环。这个循环会一直运行，直到 i 达到 max_turns (5次)。
    # 这是一个“安全阀”，防止 AI 无限循环下去。
    while i < max_turns:
        i += 1

        # --- 思考 (Think) ---
        # 获取AI返回的内容
        result = bot(next_prompt)
        print(result)

        # --- 寻找行动 (Parse Action) ---
        actions = [
            # 1. result.split('\n')：把 AI 的响应按行（\n）分割成一个列表。
            # 2. for a in ...：遍历这个列表中的每一行。
            # 3. if action_re.match(a)：检查这一行是否符合'action_re'（动作正则）定义的格式
            # 4. action_re.match(a)：如果符合，就把这个“匹配对象”存入'actions'列表。
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]

        # --- 检查是否需要行动 ---
        if actions:
            # --- 执行行动 (Act) ---
            # 去除匹配到的行动名称和行动输入
            action, action_input = actions[0].groups()
            # 检查 AI 想要的动作是否在我们预定义的'known_actions'（已知工具）中。
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            # 执行工具！
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            # 如果 'actions' 列表是空的，说明 AI 的响应中没有包含任何“Action”。
            # 这意味着 AI 认为它已经完成了任务，'result'里就是最终答案。
            return

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: To find the combined weight of the two dogs, I need to first determine the average weight of each breed and then add them together. I will use the `average_dog_weight` action for both breeds.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Thought: Now that I know the average weight of a Border Collie, I need to find the average weight of a Scottish Terrier and then add the two weights together.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Thought: Now that I have the average weights for both a Border Collie (37 lbs) and a Scottish Terrier (20 lbs), I can add these two weights together to find the combined weight of the two dogs.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: The combined weight of a Border Collie and a Scottish Terrier is 5

: 